# PARHAF corpus cleaning and publication


In [ ]:
%load_ext autoreload
%autoreload 2

from typing import Dict, List
from reports_extractor.utils import parse_patient_document, document_generator, build_json_output_header, WordCounter, InvalidPathException, DocumentParsingException
from reports_extractor.normalization import NormalizationException
from reports_extractor.logger import configure_logging
from reports_extractor.publication import publish_dataset
import logging
import os
import yaml
import json
from tqdm.auto import tqdm
import traceback


CONFIG_DIR = "config"
CONFIG_FILE = os.path.join(CONFIG_DIR, "config.yaml")

cfg = yaml.safe_load(open(CONFIG_FILE, "r") )

logdir = cfg.get("logdir", None)

configure_logging(level=logging.WARNING, logdir=logdir)
logger = logging.getLogger("main")
if logdir:
    print(f"Logs will be saved to {logdir}")

# Output paths from config
out_directory = cfg['out_directory']
out_json_file = os.path.join(out_directory, 'patients.json')

In [ ]:
# Configuration options
add_diagnostic_codes_to_structured_abstract = cfg["add_diagnostic_codes"]
count_words = cfg["add_word_count"]
if count_words:
    word_counter = WordCounter()
    logger.info("Word count will be added to each report.")
else:
    word_counter = None
overwrite = cfg.get("overwrite", False)
debug = False

patients: List[Dict] = []
patient_ids = set()

i = 0
doc_number = 0
for docx_path, doc_metadata in tqdm(document_generator(cfg)):
    # if not (doc_metadata["specialty"] == "GYNECOLOGIE" and doc_metadata["local_id"] == "00017"):
    #     continue
    i += 1
    if docx_path is None:
        continue
    # logger.info(docx_path)
    # Build a Patient object from the docx file
    # containing metadata and the reports
    # 1. Get metadata from the path
    try:
        patient = parse_patient_document(docx_path, metadata=doc_metadata, 
                                         add_diagnostic_codes_to_structured_abstract=add_diagnostic_codes_to_structured_abstract)
    except InvalidPathException as e:
        logger.error(e.message)
        continue
    except DocumentParsingException as e:
        logger.error(e.message)
        continue
    except NormalizationException as e:
        logger.error(f"Error parsing document {docx_path}: {e.message}")
        continue
    except Exception:
        logger.error(f"Error parsing document {docx_path}")
        logger.error(traceback.format_exc())
        continue
    # 2. Check uniqueness of patient ID
    if patient.get_id() in patient_ids:
        logger.error(f"Duplicate patient ID found: {patient.get_id()} (from file {docx_path}). ")
        continue
        # raise ValueError(f"Duplicate patient ID found: {patient.get_id()} (from file {docx_path}). ")
    patient_ids.add(patient.get_id())
    # 3. Get the reports from the docx content
    # Save the text of the reports into separate text files
    # Replace each report content by its path in the Patient object
    patient.extract_reports_to_files(out_directory, overwrite=overwrite, 
                                     word_counter=word_counter,
                                     one_report_per_patient=cfg.get("one_report_per_patient", False))
    patients.append(patient.to_dict())
    doc_number += len(patient.get_documents())

    if debug:
        if i >= 5:
            break

# ensure output dir exists
json_data = build_json_output_header(cfg) | {   
    "patient_count": len(patients),
    "document_count": doc_number,
    "data": patients
}
os.makedirs(os.path.dirname(out_json_file), exist_ok=True)

# convert into json
with open(out_json_file, 'w') as final:
    json.dump(json_data, final)
    logger.info(f"JSON metadata file written to {out_json_file}")
    
logger.info(f"{len(patients)} patients processed.")
if len(patients) != i:
    logger.error(f"{i - len(patients)} patients were skipped due to errors.")
    print(f"{i - len(patients)} patients were skipped due to errors.")


### Hugging Face corpus creation

In [ ]:
hf_dataset = cfg.get("hf_dataset", False)

if hf_dataset:
    publish_dataset(
        raw_data_dir=out_directory,
        json_corpus_file=out_json_file,
        cfg=cfg
    )
